# PCAP Dataset Builder (robust, parallel, stratified)

Dieses Notebook verarbeitet **alle PCAP-Dateien** in einem Ordner robust und möglichst schnell zu einem sauberen ML-Dataset.

Es ist fehlertolerant:
- kaputte oder abgeschnittene PCAPs brechen den Lauf **nicht** ab
- teilweise lesbare PCAPs werden, wenn möglich, **trotz Warnung** genutzt
- jede Datei wird separat verarbeitet und zwischengespeichert
- ein Manifest/Report dokumentiert Erfolge, Warnungen und Fehler

Es ist auf Geschwindigkeit optimiert:
- **parallele Verarbeitung** mehrerer PCAPs per `ProcessPoolExecutor`
- `tshark` als schneller Parser
- Speicherung pro Datei als **Parquet-Checkpoint**
- nur robuste, trainingsgeeignete numerische Features
- Entfernen von Identifier-/Leakage-/Redundanz-Features

Am Ende erzeugt das Notebook:
- `train_80.parquet`
- `test_20.parquet`
- optional CSV-Exporte
- `processing_report.json`
- `feature_report.json`


In [1]:
from pathlib import Path
import os, json, math, shutil, subprocess, warnings
from collections import Counter
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PCAP_DIR = Path("5G-pcap-datasets")
WORK_DIR = Path("pcap_ml_output")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
FINAL_DIR = WORK_DIR / "final"

WINDOW_PACKETS = 200
TARGET_SAMPLES_PER_FILE = 10000
TEST_SIZE = 0.20
MIN_TEST_SAMPLES_PER_CLASS = 2
SEED = 42

MISSINGNESS_THRESHOLD = 0.30
NEAR_ZERO_UNIQUE_THRESHOLD = 1
DOMINANT_VALUE_THRESHOLD = 0.995
CORRELATION_THRESHOLD = 0.98

FAST_IO_SEP = ","
READ_PARQUET_ENGINE = "pyarrow"

CPU_COUNT = os.cpu_count() or 4
MAX_WORKERS = min(8, max(2, CPU_COUNT - 1))
print(f"Detected CPUs: {CPU_COUNT}")
print(f"Using workers: {MAX_WORKERS}")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

pcap_files = sorted(
    [p for p in PCAP_DIR.glob("*.pcap")] +
    [p for p in PCAP_DIR.glob("*.pcapng")]
)
print(f"PCAP files found: {len(pcap_files)}")
for p in pcap_files[:10]:
    print(" -", p.name)

existing_checkpoints = sorted(CHECKPOINT_DIR.glob("*.parquet"))
print(f"Checkpoints found: {len(existing_checkpoints)}")
for cp in existing_checkpoints[:10]:
    print(" -", cp.name)

if len(pcap_files) == 0:
    raise FileNotFoundError(f"Keine PCAP-Dateien in {PCAP_DIR.resolve()} gefunden.")


Detected CPUs: 8
Using workers: 7
PCAP files found: 14
 - gaming_r0.pcap
 - gaming_r1.pcap
 - n2_flooding.pcap
 - n3_icmp_flooding.pcap
 - n3_tcp_flooding.pcap
 - n3_udp_flooding.pcap
 - n4_flooding.pcap
 - n6_flooding.pcap
 - ping_r0.pcap
 - ping_r1.pcap
Checkpoints found: 14
 - gaming_r0.parquet
 - gaming_r1.parquet
 - n2_flooding.parquet
 - n3_icmp_flooding.parquet
 - n3_tcp_flooding.parquet
 - n3_udp_flooding.parquet
 - n4_flooding.parquet
 - n6_flooding.parquet
 - ping_r0.parquet
 - ping_r1.parquet


## Sanity Check: Umgebung und Tools

In [2]:

def check_cmd(cmd):
    try:
        res = subprocess.run(cmd, capture_output=True, text=True)
        return res.returncode == 0, (res.stdout or res.stderr).strip()
    except Exception as e:
        return False, str(e)

ok, out = check_cmd(["tshark", "-v"])
print("tshark installed:", ok)
print(out.splitlines()[0] if out else "")

ok2, out2 = check_cmd(["editcap", "-h"])
print("editcap installed:", ok2)

disk = shutil.disk_usage(str(WORK_DIR.resolve()))
print(f"Free disk space: {disk.free / 1e9:.2f} GB")

if not ok:
    raise RuntimeError("tshark ist nicht installiert oder nicht im PATH.")


tshark installed: True
TShark (Wireshark) 4.2.2 (Git v4.2.2 packaged as 4.2.2-1.1build3).
editcap installed: True
Free disk space: 52.43 GB


## Hilfsfunktionen

In [3]:
import re
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

TSHARK_FIELDS = [
    "frame.time_epoch",
    "frame.len",
    "ip.proto",
    "tcp.srcport",
    "tcp.dstport",
    "udp.srcport",
    "udp.dstport",
    "tcp.flags.syn",
    "tcp.flags.ack",
    "tcp.flags.fin",
    "tcp.flags.reset",
    "tcp.flags.push",
    "tcp.flags.urg",
    "icmp.type",
]

FINAL_FEATURE_DROP_HINTS = {
    "src_ip", "dst_ip", "timestamp", "frame.time", "frame.time_epoch",
    "src_port", "dst_port", "tcp.srcport", "tcp.dstport", "udp.srcport", "udp.dstport",
    "file_name", "label_full", "label_family"
}

TSHARK_DTYPES = {
    "frame.time_epoch": "float64",
    "frame.len": "float64",
    "ip.proto": "float64",
    "tcp.srcport": "float64",
    "tcp.dstport": "float64",
    "udp.srcport": "float64",
    "udp.dstport": "float64",
    "tcp.flags.syn": "float64",
    "tcp.flags.ack": "float64",
    "tcp.flags.fin": "float64",
    "tcp.flags.reset": "float64",
    "tcp.flags.push": "float64",
    "tcp.flags.urg": "float64",
    "icmp.type": "float64",
}

def parse_labels_from_filename(filename: str):
    stem = Path(filename).stem
    parts = stem.split("_")
    if len(parts) >= 3:
        label_family = "_".join(parts[:-1])
    else:
        label_family = stem
    label_full = stem
    return label_full, label_family

def make_tempdir():
    for base in ["/tmp", "/var/tmp", str(WORK_DIR.resolve())]:
        p = Path(base)
        try:
            p.mkdir(parents=True, exist_ok=True)
            test = p / ".write_test"
            test.write_text("ok")
            test.unlink()
            return str(p)
        except Exception:
            continue
    return str(WORK_DIR.resolve())

SAFE_TMPDIR = make_tempdir()
os.environ["TMPDIR"] = SAFE_TMPDIR

def tshark_cmd_for_file(path_str: str):
    cmd = ["tshark", "-n", "-r", path_str, "-T", "fields"]
    for f in TSHARK_FIELDS:
        cmd += ["-e", f]
    # Use a plain comma separator. All extracted fields are numeric, so commas are safe and robust.
    cmd += ["-E", "header=n", "-E", f"separator={FAST_IO_SEP}", "-E", "quote=n", "-E", "occurrence=f"]
    return cmd

def try_repair_pcap(input_path: Path, repaired_path: Path):
    cmd = ["editcap", "-F", "pcap", str(input_path), str(repaired_path)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    return res.returncode == 0, (res.stdout or "") + (res.stderr or "")

def _read_tshark_bytes(stdout_bytes: bytes):
    if not stdout_bytes or not stdout_bytes.strip():
        return pd.DataFrame(columns=TSHARK_FIELDS)
    bio = BytesIO(stdout_bytes)
    df = pd.read_csv(
        bio,
        sep=FAST_IO_SEP,
        names=TSHARK_FIELDS,
        header=None,
        dtype=TSHARK_DTYPES,
        engine="c",
        na_values=["", "nan", "NaN"],
        keep_default_na=True,
        low_memory=False,
    )
    return df

def tshark_extract_to_dataframe(pcap_path: Path):
    res = subprocess.run(
        tshark_cmd_for_file(str(pcap_path)),
        capture_output=True,
        text=False,
        env={**os.environ, "TMPDIR": SAFE_TMPDIR},
    )

    stderr = (res.stderr or b"").decode("utf-8", errors="replace").strip()
    stdout = res.stdout or b""

    if stdout.strip():
        df = _read_tshark_bytes(stdout)
        status = "ok" if res.returncode == 0 else "partial"
        return df, status, stderr

    repaired_path = Path(SAFE_TMPDIR) / f"{pcap_path.stem}_repaired.pcap"
    repaired_ok, repair_log = try_repair_pcap(pcap_path, repaired_path)
    if repaired_ok and repaired_path.exists():
        res2 = subprocess.run(
            tshark_cmd_for_file(str(repaired_path)),
            capture_output=True,
            text=False,
            env={**os.environ, "TMPDIR": SAFE_TMPDIR},
        )
        stdout2 = res2.stdout or b""
        stderr2 = (
            ((res2.stderr or b"").decode("utf-8", errors="replace")).strip()
            + "\n"
            + repair_log
        ).strip()
        if stdout2.strip():
            df = _read_tshark_bytes(stdout2)
            return df, "repaired", stderr2

    raise RuntimeError(stderr[:2000] if stderr else "Unbekannter tshark/editcap Fehler")

def coerce_numeric(df: pd.DataFrame):
    return df.apply(pd.to_numeric, errors="coerce")

def build_window_features(pkt_df: pd.DataFrame, window_packets: int, label_full: str, label_family: str, file_name: str):
    if pkt_df is None or pkt_df.empty:
        return pd.DataFrame()

    rename_map = {
        "frame.time_epoch": "ts",
        "frame.len": "pkt_len",
        "ip.proto": "ip_proto",
        "tcp.srcport": "tcp_sport",
        "tcp.dstport": "tcp_dport",
        "udp.srcport": "udp_sport",
        "udp.dstport": "udp_dport",
        "tcp.flags.syn": "syn",
        "tcp.flags.ack": "ack",
        "tcp.flags.fin": "fin",
        "tcp.flags.reset": "rst",
        "tcp.flags.push": "psh",
        "tcp.flags.urg": "urg",
        "icmp.type": "icmp_type",
    }

    df = pkt_df.rename(columns=rename_map, copy=True)
    df = coerce_numeric(df)

    n = len(df)
    n_full = (n // window_packets) * window_packets
    if n_full < window_packets:
        return pd.DataFrame()

    df = df.iloc[:n_full].copy()
    df["l4_sport"] = df["tcp_sport"].combine_first(df["udp_sport"])
    df["l4_dport"] = df["tcp_dport"].combine_first(df["udp_dport"])

    df["iat"] = df["ts"].diff().clip(lower=0)
    df["is_tcp"] = (df["ip_proto"] == 6).astype("float64")
    df["is_udp"] = (df["ip_proto"] == 17).astype("float64")
    df["is_icmp"] = (df["ip_proto"] == 1).astype("float64")
    df["sport_missing"] = df["l4_sport"].isna().astype("float64")
    df["dport_missing"] = df["l4_dport"].isna().astype("float64")
    df["sport_low"] = (df["l4_sport"] < 1024).fillna(False).astype("float64")
    df["dport_low"] = (df["l4_dport"] < 1024).fillna(False).astype("float64")
    df["sport_high"] = (df["l4_sport"] >= 49152).fillna(False).astype("float64")
    df["dport_high"] = (df["l4_dport"] >= 49152).fillna(False).astype("float64")
    df["window_id"] = np.arange(n_full, dtype=np.int64) // window_packets

    grp = df.groupby("window_id", sort=True, observed=True)

    pkt_len_stats = grp["pkt_len"].agg(["mean", "std", "min", "max", "median", "sum"])
    pkt_len_stats = pkt_len_stats.rename(columns={
        "mean": "pkt_len_mean",
        "std": "pkt_len_std",
        "min": "pkt_len_min",
        "max": "pkt_len_max",
        "median": "pkt_len_median",
        "sum": "pkt_len_sum",
    })
    pkt_len_stats["pkt_len_std"] = grp["pkt_len"].std(ddof=0)
    pkt_len_q = grp["pkt_len"].quantile([0.25, 0.75]).unstack(level=-1)
    pkt_len_q = pkt_len_q.rename(columns={0.25: "pkt_len_q25", 0.75: "pkt_len_q75"})

    iat_stats = grp["iat"].agg(["mean", "min", "max", "median"])
    iat_stats = iat_stats.rename(columns={
        "mean": "iat_mean",
        "min": "iat_min",
        "max": "iat_max",
        "median": "iat_median",
    })
    iat_stats["iat_std"] = grp["iat"].std(ddof=0)

    ratio_base = df[[
        "window_id", "is_tcp", "is_udp", "is_icmp", "sport_missing", "dport_missing",
        "sport_low", "dport_low", "sport_high", "dport_high"
    ]].copy()
    for _flag in ["syn", "ack", "fin", "rst", "psh", "urg"]:
        ratio_base[_flag] = df[_flag].fillna(0.0)
    ratio_stats = ratio_base.groupby("window_id", sort=True, observed=True)[[
        "is_tcp", "is_udp", "is_icmp", "syn", "ack", "fin", "rst", "psh", "urg",
        "sport_missing", "dport_missing", "sport_low", "dport_low", "sport_high", "dport_high"
    ]].mean().rename(columns={
        "is_tcp": "tcp_ratio",
        "is_udp": "udp_ratio",
        "is_icmp": "icmp_ratio",
        "syn": "syn_ratio",
        "ack": "ack_ratio",
        "fin": "fin_ratio",
        "rst": "rst_ratio",
        "psh": "psh_ratio",
        "urg": "urg_ratio",
        "sport_missing": "sport_missing_ratio",
        "dport_missing": "dport_missing_ratio",
        "sport_low": "sport_low_ratio",
        "dport_low": "dport_low_ratio",
        "sport_high": "sport_high_ratio",
        "dport_high": "dport_high_ratio",
    })

    misc_stats = pd.DataFrame({
        "packets_per_window": grp.size().astype("float64"),
        "flow_duration_window": (grp["ts"].max() - grp["ts"].min()).astype("float64"),
        "unique_proto_count": grp["ip_proto"].nunique(dropna=True).astype("float64"),
        "unique_icmp_type_count": grp["icmp_type"].nunique(dropna=True).astype("float64"),
    })

    feat_df = pd.concat([pkt_len_stats, pkt_len_q, iat_stats, ratio_stats, misc_stats], axis=1).reset_index(drop=True)
    feat_df["label_full"] = label_full
    feat_df["label_family"] = label_family
    feat_df["file_name"] = file_name

    ordered_cols = [
        "packets_per_window",
        "pkt_len_mean", "pkt_len_std", "pkt_len_min", "pkt_len_max", "pkt_len_median", "pkt_len_q25", "pkt_len_q75", "pkt_len_sum",
        "iat_mean", "iat_std", "iat_min", "iat_max", "iat_median",
        "flow_duration_window",
        "tcp_ratio", "udp_ratio", "icmp_ratio",
        "syn_ratio", "ack_ratio", "fin_ratio", "rst_ratio", "psh_ratio", "urg_ratio",
        "sport_missing_ratio", "dport_missing_ratio",
        "sport_low_ratio", "dport_low_ratio", "sport_high_ratio", "dport_high_ratio",
        "unique_proto_count", "unique_icmp_type_count",
        "label_full", "label_family", "file_name",
    ]
    return feat_df[ordered_cols]

def per_file_target_sample(n_rows, target=TARGET_SAMPLES_PER_FILE):
    return min(target, n_rows)

def process_one_pcap(pcap_path_str: str):
    pcap_path = Path(pcap_path_str)
    file_name = pcap_path.name
    label_full, label_family = parse_labels_from_filename(file_name)
    out_path = CHECKPOINT_DIR / f"{pcap_path.stem}.parquet"

    if out_path.exists():
        try:
            df = pd.read_parquet(out_path, engine=READ_PARQUET_ENGINE)
            return {
                "file": file_name,
                "status": "cached",
                "packets_extracted": None,
                "windows_created": int(len(df)),
                "label_full": label_full,
                "label_family": label_family,
                "checkpoint": str(out_path),
                "warning": ""
            }
        except Exception as e:
            print(f"[CACHE-READ-FAIL] {out_path.name}: {type(e).__name__}: {e}", flush=True)

    try:
        pkt_df, extract_status, msg = tshark_extract_to_dataframe(pcap_path)
        feat_df = build_window_features(
            pkt_df=pkt_df,
            window_packets=WINDOW_PACKETS,
            label_full=label_full,
            label_family=label_family,
            file_name=file_name,
        )

        if feat_df is None or feat_df.empty:
            packet_count = int(len(pkt_df)) if pkt_df is not None else 0
            return {
                "file": file_name,
                "status": "empty",
                "packets_extracted": packet_count,
                "windows_created": 0,
                "label_full": label_full,
                "label_family": label_family,
                "checkpoint": None,
                "warning": (
                    f"Nur {packet_count} Pakete extrahiert; WINDOW_PACKETS={WINDOW_PACKETS} erzeugt 0 volle Fenster. "
                    + (msg[:400] if msg else "")
                )[:800]
            }

        target_n = per_file_target_sample(len(feat_df))
        if len(feat_df) > target_n:
            feat_df = feat_df.sample(n=target_n, random_state=SEED)

        tmp_out_path = out_path.with_suffix(".tmp.parquet")
        feat_df.to_parquet(tmp_out_path, index=False, engine=READ_PARQUET_ENGINE)
        tmp_out_path.replace(out_path)

        return {
            "file": file_name,
            "status": extract_status,
            "packets_extracted": int(len(pkt_df)),
            "windows_created": int(len(feat_df)),
            "label_full": label_full,
            "label_family": label_family,
            "checkpoint": str(out_path),
            "warning": msg[:500] if msg else ""
        }
    except Exception as e:
        return {
            "file": file_name,
            "status": "failed",
            "packets_extracted": 0,
            "windows_created": 0,
            "label_full": label_full,
            "label_family": label_family,
            "checkpoint": None,
            "warning": str(e)[:1000]
        }


## Parallel verarbeiten und Checkpoints schreiben

In [4]:
processing_summary = []

print(f"Starte {len(pcap_files)} PCAPs mit {MAX_WORKERS} Workern...", flush=True)
for p in pcap_files:
    print(f"Geplant: {p.name}", flush=True)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_one_pcap, str(p)): p.name for p in pcap_files}

    for i, fut in enumerate(as_completed(futures), 1):
        fname = futures[fut]
        try:
            res = fut.result()
        except Exception as e:
            res = {
                "file": fname,
                "status": "failed",
                "packets_extracted": 0,
                "windows_created": 0,
                "label_full": parse_labels_from_filename(fname)[0],
                "label_family": parse_labels_from_filename(fname)[1],
                "checkpoint": None,
                "warning": f"{type(e).__name__}: {e}",
            }
            print(f"[{i}/{len(futures)}] {fname} -> failed | {type(e).__name__}: {e}", flush=True)
        else:
            print(
                f"[{i}/{len(futures)}] {res['file']} -> {res['status']} | windows={res.get('windows_created', 0)}",
                flush=True
            )

        processing_summary.append(res)

summary_df = pd.DataFrame(processing_summary).sort_values(["status", "file"]).reset_index(drop=True)
summary_df

Starte 14 PCAPs mit 7 Workern...
Geplant: gaming_r0.pcap
Geplant: gaming_r1.pcap
Geplant: n2_flooding.pcap
Geplant: n3_icmp_flooding.pcap
Geplant: n3_tcp_flooding.pcap
Geplant: n3_udp_flooding.pcap
Geplant: n4_flooding.pcap
Geplant: n6_flooding.pcap
Geplant: ping_r0.pcap
Geplant: ping_r1.pcap
Geplant: streaming_r0.pcap
Geplant: streaming_r1.pcap
Geplant: webbrowsing_r0.pcap
Geplant: webbrowsing_r1.pcap
[1/14] n4_flooding.pcap -> cached | windows=8809
[2/14] n3_tcp_flooding.pcap -> cached | windows=7372
[3/14] n3_udp_flooding.pcap -> cached | windows=10000
[4/14] gaming_r0.pcap -> cached | windows=8968
[5/14] n2_flooding.pcap -> cached | windows=10000
[6/14] gaming_r1.pcap -> cached | windows=8952
[7/14] n3_icmp_flooding.pcap -> cached | windows=10000
[8/14] ping_r0.pcap -> cached | windows=214
[9/14] n6_flooding.pcap -> cached | windows=10000
[10/14] webbrowsing_r1.pcap -> cached | windows=3
[11/14] ping_r1.pcap -> cached | windows=191
[12/14] streaming_r0.pcap -> cached | windows=1000

,file,status,packets_extracted,windows_created,label_full,label_family,checkpoint,warning
0,gaming_r0.pcap,cached,None,8968,gaming_r0,gaming_r0,pcap_ml_output/checkpoints/gaming_r0.parquet,
1,gaming_r1.pcap,cached,None,8952,gaming_r1,gaming_r1,pcap_ml_output/checkpoints/gaming_r1.parquet,
2,n2_flooding.pcap,cached,None,10000,n2_flooding,n2_flooding,pcap_ml_output/checkpoints/n2_flooding.parquet,
3,n3_icmp_flooding.pcap,cached,None,10000,n3_icmp_flooding,n3_icmp,pcap_ml_output/checkpoints/n3_icmp_flooding.pa...,
4,n3_tcp_flooding.pcap,cached,None,7372,n3_tcp_flooding,n3_tcp,pcap_ml_output/checkpoints/n3_tcp_flooding.par...,
5,n3_udp_flooding.pcap,cached,None,10000,n3_udp_flooding,n3_udp,pcap_ml_output/checkpoints/n3_udp_flooding.par...,
6,n4_flooding.pcap,cached,None,8809,n4_flooding,n4_flooding,pcap_ml_output/checkpoints/n4_flooding.parquet,
7,n6_flooding.pcap,cached,None,10000,n6_flooding,n6_flooding,pcap_ml_output/checkpoints/n6_flooding.parquet,
8,ping_r0.pcap,cached,None,214,ping_r0,ping_r0,pcap_ml_output/checkpoints/ping_r0.parquet,
9,ping_r1.pcap,cached,None,191,ping_r1,ping_r1,pcap_ml_output/checkpoints/ping_r1.parquet,


## Sanity Check nach Verarbeitung

In [5]:

ok_df = summary_df[summary_df["checkpoint"].notna()].copy()
failed_df = summary_df[summary_df["status"].eq("failed")].copy()

print("Erfolgreich/verwendbar:", len(ok_df))
print("Fehlgeschlagen:", len(failed_df))
if len(failed_df):
    display(failed_df[["file", "status", "warning"]])

if len(ok_df) == 0:
    raise RuntimeError("Keine verwertbaren PCAPs verarbeitet.")

assert (ok_df["windows_created"] > 0).all(), "Mindestens eine Datei hat 0 Fenster."
assert (ok_df["windows_created"] <= TARGET_SAMPLES_PER_FILE).all(), "Mindestens eine Datei überschreitet target samples."

schemas = []
for cp in ok_df["checkpoint"]:
    cols = tuple(pd.read_parquet(cp).columns.tolist())
    schemas.append(cols)

same_schema = all(s == schemas[0] for s in schemas)
print("Alle Checkpoints haben identisches Schema:", same_schema)
if not same_schema:
    raise RuntimeError("Nicht alle Checkpoints haben dieselben Features.")


Erfolgreich/verwendbar: 14
Fehlgeschlagen: 0
Alle Checkpoints haben identisches Schema: True


## Zusammenführen

In [6]:
frames = [pd.read_parquet(cp, engine=READ_PARQUET_ENGINE) for cp in ok_df["checkpoint"]]
full_df = pd.concat(frames, ignore_index=True, copy=False)
print(full_df.shape)
full_df.head()

(94532, 35)


,packets_per_window,pkt_len_mean,pkt_len_std,pkt_len_min,pkt_len_max,pkt_len_median,pkt_len_q25,pkt_len_q75,pkt_len_sum,iat_mean,...,dport_missing_ratio,sport_low_ratio,dport_low_ratio,sport_high_ratio,dport_high_ratio,unique_proto_count,unique_icmp_type_count,label_full,label_family,file_name
0,200.0,116.57,9.213311,100.0,124.0,124.0,112.0,124.0,23314.0,0.638093,...,0.18,0.0,0.0,0.0,0.0,2.0,0.0,gaming_r0,gaming_r0,gaming_r0.pcap
1,200.0,120.28,5.549919,112.0,124.0,124.0,112.0,124.0,24056.0,0.008316,...,0.00,0.0,0.0,0.0,0.0,1.0,0.0,gaming_r0,gaming_r0,gaming_r0.pcap
2,200.0,120.76,5.327513,112.0,124.0,124.0,112.0,124.0,24152.0,0.008347,...,0.00,0.0,0.0,0.0,0.0,1.0,0.0,gaming_r0,gaming_r0,gaming_r0.pcap
3,200.0,120.76,5.327513,112.0,124.0,124.0,112.0,124.0,24152.0,0.009789,...,0.00,0.0,0.0,0.0,0.0,1.0,0.0,gaming_r0,gaming_r0,gaming_r0.pcap
4,200.0,120.16,5.597714,112.0,124.0,124.0,112.0,124.0,24032.0,0.006502,...,0.00,0.0,0.0,0.0,0.0,1.0,0.0,gaming_r0,gaming_r0,gaming_r0.pcap


## Feature Cleaning

In [7]:
meta_cols = ["label_full", "label_family", "file_name"]
feature_cols = [c for c in full_df.columns if c not in meta_cols]

feature_report = {
    "initial_feature_count": len(feature_cols),
    "dropped_identifier_like": [],
    "dropped_missingness": [],
    "dropped_constant_or_single_unique": [],
    "dropped_dominant_value": [],
    "dropped_duplicate_columns": [],
    "dropped_high_correlation": [],
}

X = full_df[feature_cols].copy()

drop_id = [c for c in X.columns if c in FINAL_FEATURE_DROP_HINTS or c.endswith("_port") or c.endswith("_ip")]
feature_report["dropped_identifier_like"] = drop_id
X = X.drop(columns=drop_id, errors="ignore")

for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")

missing_rate = X.isna().mean()
drop_missing = missing_rate[missing_rate > MISSINGNESS_THRESHOLD].index.tolist()
feature_report["dropped_missingness"] = drop_missing
X = X.drop(columns=drop_missing, errors="ignore")

single_unique = [c for c in X.columns if X[c].nunique(dropna=True) <= NEAR_ZERO_UNIQUE_THRESHOLD]
feature_report["dropped_constant_or_single_unique"] = single_unique
X = X.drop(columns=single_unique, errors="ignore")

drop_dom = []
for c in X.columns:
    vc = X[c].value_counts(dropna=False, normalize=True)
    if len(vc) and vc.iloc[0] >= DOMINANT_VALUE_THRESHOLD:
        drop_dom.append(c)
feature_report["dropped_dominant_value"] = drop_dom
X = X.drop(columns=drop_dom, errors="ignore")

dup_mask = X.T.duplicated()
drop_dup = X.columns[dup_mask].tolist()
feature_report["dropped_duplicate_columns"] = drop_dup
X = X.loc[:, ~dup_mask].copy()

if X.shape[1] == 0:
    raise RuntimeError("Nach Feature-Bereinigung sind keine Features mehr übrig.")

# Robust imputation without requiring a separate sklearn step.
X_imp = X.copy()
column_medians = X_imp.median(numeric_only=True)
X_imp = X_imp.fillna(column_medians)
X_imp = X_imp.fillna(0.0)

# Remove highly correlated features on the imputed matrix.
if X_imp.shape[1] <= 1:
    drop_corr = []
else:
    corr_matrix = X_imp.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    drop_corr = [column for column in upper.columns if (upper[column] > CORRELATION_THRESHOLD).any()]

feature_report["dropped_high_correlation"] = drop_corr
X_final = X_imp.drop(columns=drop_corr, errors="ignore")

clean_df = pd.concat([X_final.reset_index(drop=True), full_df[meta_cols].reset_index(drop=True)], axis=1)

feature_report["final_feature_count"] = len([c for c in clean_df.columns if c not in meta_cols])

print("Initial features:", feature_report["initial_feature_count"])
print("Final features:", feature_report["final_feature_count"])
feature_report


Initial features: 32
Final features: 18


{'initial_feature_count': 32,
 'dropped_identifier_like': [],
 'dropped_missingness': [],
 'dropped_constant_or_single_unique': ['packets_per_window', 'urg_ratio'],
 'dropped_dominant_value': ['fin_ratio',
  'rst_ratio',
  'sport_low_ratio',
  'dport_high_ratio'],
 'dropped_duplicate_columns': ['dport_missing_ratio'],
 'dropped_high_correlation': ['pkt_len_max',
  'pkt_len_q25',
  'pkt_len_q75',
  'pkt_len_sum',
  'flow_duration_window',
  'psh_ratio',
  'dport_low_ratio'],
 'final_feature_count': 18}

## Stratified 80/20 Split

In [8]:
import math

class_counts = clean_df["label_full"].value_counts().sort_index()
print("Class counts before split:")
print(class_counts)

if (class_counts < 2).any():
    too_small = class_counts[class_counts < 2]
    raise RuntimeError(
        "Mindestens eine Klasse hat <2 Samples und kann nicht gleichzeitig in Train und Test sein: "
        + str(too_small.to_dict())
    )

split_source_df = clean_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True).copy()

requested_min_test = int(MIN_TEST_SAMPLES_PER_CLASS)
if requested_min_test < 1:
    raise ValueError("MIN_TEST_SAMPLES_PER_CLASS muss >= 1 sein.")

requested_test_abs = int(math.ceil(len(split_source_df) * TEST_SIZE)) if isinstance(TEST_SIZE, float) else int(TEST_SIZE)
if requested_test_abs <= 0:
    raise ValueError("TEST_SIZE ist zu klein und ergibt 0 Test-Samples.")

per_class_test = {}
for label, count in class_counts.items():
    proportional = int(math.ceil(count * TEST_SIZE)) if isinstance(TEST_SIZE, float) else int(round(count * (requested_test_abs / len(split_source_df))))
    n_test = max(requested_min_test, proportional)
    n_test = min(n_test, int(count) - 1)  # mindestens 1 Sample muss fuer Train bleiben
    if n_test < 1:
        raise RuntimeError(f"Klasse {label} kann nicht gesplittet werden.")
    per_class_test[label] = n_test

actual_test_abs = sum(per_class_test.values())
actual_test_ratio = actual_test_abs / len(split_source_df)

print("\nRequested TEST_SIZE:", TEST_SIZE)
print("Requested minimum test samples per class:", requested_min_test)
print("Actual test rows from per-class allocation:", actual_test_abs)
print(f"Actual test ratio: {actual_test_ratio:.4f}")

test_parts = []
train_parts = []

for label, grp in split_source_df.groupby("label_full", sort=True):
    grp = grp.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n_test = per_class_test[label]
    test_part = grp.iloc[:n_test].copy()
    train_part = grp.iloc[n_test:].copy()
    test_parts.append(test_part)
    train_parts.append(train_part)

test_df = pd.concat(test_parts, ignore_index=True)
train_df = pd.concat(train_parts, ignore_index=True)

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("\nSplit source shape:", split_source_df.shape)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_counts = train_df["label_full"].value_counts().sort_index()
test_counts = test_df["label_full"].value_counts().sort_index()

print("\nTrain counts:")
print(train_counts)
print("\nTest counts:")
print(test_counts)

missing_train = sorted(set(split_source_df["label_full"]) - set(train_df["label_full"]))
missing_test = sorted(set(split_source_df["label_full"]) - set(test_df["label_full"]))
assert not missing_train, f"Fehlende Klassen in train: {missing_train}"
assert not missing_test, f"Fehlende Klassen in test: {missing_test}"

too_small_test = test_counts[test_counts < requested_min_test]
if len(too_small_test):
    raise RuntimeError(
        "Mindestens eine Klasse hat weniger Test-Samples als gewuenscht: "
        + str(too_small_test.to_dict())
    )

Class counts before split:
label_full
gaming_r0            8968
gaming_r1            8952
n2_flooding         10000
n3_icmp_flooding    10000
n3_tcp_flooding      7372
n3_udp_flooding     10000
n4_flooding          8809
n6_flooding         10000
ping_r0               214
ping_r1               191
streaming_r0        10000
streaming_r1        10000
webbrowsing_r0         23
webbrowsing_r1          3
Name: count, dtype: int64

Requested TEST_SIZE: 0.2
Requested minimum test samples per class: 2
Actual test rows from per-class allocation: 18911
Actual test ratio: 0.2000

Split source shape: (94532, 21)
Train shape: (75621, 21)
Test shape: (18911, 21)

Train counts:
label_full
gaming_r0           7174
gaming_r1           7161
n2_flooding         8000
n3_icmp_flooding    8000
n3_tcp_flooding     5897
n3_udp_flooding     8000
n4_flooding         7047
n6_flooding         8000
ping_r0              171
ping_r1              152
streaming_r0        8000
streaming_r1        8000
webbrowsing_r0    

## Speichern

In [9]:
train_parquet = FINAL_DIR / "train_80.parquet"
test_parquet = FINAL_DIR / "test_20.parquet"
full_parquet = FINAL_DIR / "full_clean.parquet"

train_csv = FINAL_DIR / "train_80.csv"
test_csv = FINAL_DIR / "test_20.csv"

train_df.to_parquet(train_parquet, index=False)
test_df.to_parquet(test_parquet, index=False)
split_source_df.to_parquet(full_parquet, index=False)
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

report = {
    "pcap_dir": str(PCAP_DIR.resolve()),
    "n_input_pcaps": int(len(pcap_files)),
    "n_successful_or_used_pcaps": int(len(ok_df)),
    "n_failed_pcaps": int(len(failed_df)),
    "workers": int(MAX_WORKERS),
    "window_packets": int(WINDOW_PACKETS),
    "target_samples_per_file": int(TARGET_SAMPLES_PER_FILE),
    "test_size_requested": TEST_SIZE,
    "min_test_samples_per_class": int(MIN_TEST_SAMPLES_PER_CLASS),
    "full_rows": int(len(split_source_df)),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "classes": sorted(split_source_df["label_full"].unique().tolist()),
    "train_counts": train_counts.to_dict(),
    "test_counts": test_counts.to_dict(),
    "failed_files": failed_df[["file", "warning"]].to_dict(orient="records"),
}

(WORK_DIR / "processing_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
(WORK_DIR / "feature_report.json").write_text(json.dumps(feature_report, indent=2, ensure_ascii=False))
summary_df.to_csv(WORK_DIR / "processing_summary.csv", index=False)

print("Gespeichert in:", FINAL_DIR.resolve())
print("Reports:", (WORK_DIR / "processing_report.json").resolve(), (WORK_DIR / "feature_report.json").resolve())

Gespeichert in: /home/ubuntu/5g-nidd/cedi-nidd/pcap_ml_output/final
Reports: /home/ubuntu/5g-nidd/cedi-nidd/pcap_ml_output/processing_report.json /home/ubuntu/5g-nidd/cedi-nidd/pcap_ml_output/feature_report.json
